# 01 — Convergence: tree and Monte Carlo vs the closed form

Two of the three engines in this repo are approximations, so the first thing to
establish is that they converge to the thing we can compute exactly.

* **Binomial tree** — a lattice approximation. Every extra step halves the
  spacing, so the price error should fall like `1/steps` (with a saw-tooth,
  because strikes rarely land on nodes).
* **Monte Carlo** — an average of random payoffs. Its error falls like
  `1/sqrt(paths)`, which is much slower: to halve the error you need 4x the paths.

The point of this notebook is to see both convergence rates and to learn which
engine to reach for: the tree when you need early exercise, Monte Carlo when the
payoff is high-dimensional (and never to rely on MC for a fourth decimal place).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from optlab.pricing.black_scholes import bs_price, bs_greeks
from optlab.pricing.binomial import binomial_price
from optlab.pricing.monte_carlo import mc_price

## 1. The tree converges to Black-Scholes

Fix an at-the-money one-year option: `S = K = 100`, `r = 5%`, `sigma = 20%`.
Black-Scholes gives about **10.4506**. Sweep the step count and plot the absolute
error on a log-log grid: a straight line with slope -1 is the `1/steps` signature.

In [ ]:
S = K = 100.0
T, r, sigma = 1.0, 0.05, 0.20

analytic = bs_price(S, K, T, r, sigma)
steps = [5, 10, 25, 50, 100, 200, 500, 1000, 2000, 5000]
tree_prices = [binomial_price(S, K, T, r, sigma, steps=n) for n in steps]
errors = [abs(p - analytic) for p in tree_prices]

print(f"Black-Scholes: {analytic:.8f}")
pd.DataFrame({"steps": steps, "tree": tree_prices, "abs_error": errors}).round(6)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog(steps, errors, "o-", label="|tree - Black-Scholes|")
ax.loglog(steps, [errors[0] * steps[0] / n for n in steps], "k--", label=r"$\propto 1/\text{steps}$")
ax.set_xlabel("tree steps")
ax.set_ylabel("absolute price error")
ax.set_title("CRR tree error decays like 1/steps")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Monte Carlo converges much more slowly

Ten thousand paths already gets inside a couple of cents, but look at the shape:
error falls like `1/sqrt(N)`. The `stderr` reported by `mc_price` is an honest
statement of that uncertainty, and the 95% interval should contain the analytic
price — that is the actual test the test suite runs.

In [ ]:
rows = []
for n_paths in [1_000, 4_000, 16_000, 64_000, 256_000]:
    res = mc_price(S, K, T, r, sigma, n_paths=n_paths, seed=2026)
    rows.append({
        "paths": n_paths,
        "price": res.price,
        "stderr": res.stderr,
        "abs_error": abs(res.price - analytic),
        "covers_BS": res.ci_low < analytic < res.ci_high,
    })
pd.DataFrame(rows).round(5)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
rows = pd.DataFrame(rows)
ax.loglog(rows["paths"], rows["stderr"], "o-", label="reported standard error")
ax.loglog(rows["paths"], rows["stderr"].iloc[0] * np.sqrt(rows["paths"].iloc[0] / rows["paths"]),
          "k--", label=r"$\propto 1/\sqrt{N}$")
ax.set_xlabel("simulated paths")
ax.set_ylabel("standard error")
ax.set_title("Monte Carlo error decays like 1/sqrt(N)")
ax.legend()
plt.tight_layout()
plt.show()

## 3. American options: the tree earns its keep

Black-Scholes has no answer for early exercise. The tree does, and the American
put premium is the price of that optionality. With a non-dividend-paying call the
premium must be zero — a free correctness check on the early-exercise branch.

In [ ]:
rows = []
for moneyness in [0.8, 0.9, 1.0, 1.1, 1.2]:
    strike = moneyness * S
    euro = binomial_price(S, strike, T, r, sigma, steps=800, kind="put")
    amer = binomial_price(S, strike, T, r, sigma, steps=800, kind="put", american=True)
    call_prem = (binomial_price(S, strike, T, r, sigma, steps=800, american=True)
                 - binomial_price(S, strike, T, r, sigma, steps=800, american=False))
    rows.append({"K": strike, "european_put": euro, "american_put": amer,
                 "early_exercise_premium": amer - euro, "american_call_premium": call_prem})
pd.DataFrame(rows).round(4)

## Takeaways

* The tree is a *consistent* approximation: refine it and it converges to the closed form.
* Monte Carlo is unbiased but noisy, and its error shrinks painfully slowly.
  Variance reduction (antithetic + control variate) buys a constant factor, not a
  better rate — see `mc_price(..., antithetic=False, control_variate=False)`.
* The American call premium is zero without dividends, which is a cheap way to
  catch a broken early-exercise condition.